In [19]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from problem import Problem
from data import DataGen
from models import * 

import matplotlib.pyplot as plt 

%load_ext autoreload
%autoreload 2

def generate_samples(n_samples, means, rho): 
    noises = torch.rand(n_samples, means.shape[0]) * rho
    return (torch.tensor(means) * ( 1 + noises)).detach().numpy()

def eval_model(forecast, X_test, Y_test, n_samples = 10):
    all_costs = [] 
    all_decisions = []

    saa_rho = 0.1
    saa_n_samples = n_samples

    if n_samples == 0: 
        saa_rho = 0 
        saa_n_samples = 1

    t = 0
    for x,d in zip(X_test, Y_test): 
        if t > 300: break
        t += 1
        
        pred = forecast(x)
        samples = generate_samples(saa_n_samples, pred, saa_rho)
        decision, _ = saa(samples, H, B, cross_costs, test=False) 
        # print(decision, samples)
        _, cost = saa(d.unsqueeze(0).detach().numpy(), H, B, cross_costs, decision[0].detach().numpy(), test=True)
        all_costs.append(cost)
        print(t, " ", np.mean(all_costs))
        all_decisions.append(decision[0].numpy())
    all_decisions = np.array(all_decisions)
    all_decisions = torch.tensor(all_decisions) 
    return all_costs, all_decisions


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [20]:
np.random.seed(0)
torch.manual_seed(0)

# make random distances
n_nodes = 2
n_features = 10

H = torch.tensor([1 for i in range(n_nodes)])
B = torch.tensor([10 for i in range(n_nodes)])  

problem_ = Problem(H, B, n_nodes)
cross_costs = problem_.cross_costs

n_data = 100
n_test = 1000
data_generator = DataGen(n_features, n_nodes)
X_train, Y_train, X_test, Y_test = data_generator.get_test_train(n_data, n_test)

In [29]:
print("Train Two-Stage -------------------------------------")
two_stage_forecast = train_two_stage(problem_, X_train, Y_train, X_test, Y_test)

Train Two-Stage -------------------------------------
epoch  0 test cost:  8.657375849026398 train cost:  7.218302576602539
epoch  10 test cost:  7.780008721652076 train cost:  6.426385322953466
epoch  20 test cost:  6.669311012159619 train cost:  5.368583422171038
epoch  30 test cost:  5.567437510848111 train cost:  4.257192551301719
epoch  40 test cost:  4.616967088097584 train cost:  3.285397975677231
epoch  50 test cost:  3.8753948302859818 train cost:  2.5715076556676255
epoch  60 test cost:  3.3426481862992694 train cost:  2.078457176866297
epoch  70 test cost:  2.9512023024694303 train cost:  1.7130972912473568
epoch  80 test cost:  2.6712173577764258 train cost:  1.4615348386345064
epoch  90 test cost:  2.384402915942019 train cost:  1.2173574938082465
epoch  100 test cost:  2.1991284711426715 train cost:  1.0620033001671367
epoch  110 test cost:  2.0764652871977844 train cost:  0.9700177961404575
epoch  120 test cost:  1.9818223176882275 train cost:  0.9012985830518292
epoch  

In [31]:
print("Train end-to-end -------------------------------------")
task_forecast = train_task_loss(problem_, X_train, Y_train, X_test, Y_test, two_stage_forecast, EPOCHS=100)

Train end-to-end -------------------------------------
epoch: 0
Cost:  0.7975223847885782
epoch  0 test cost:  1.6145834183786756 train cost:  0.7946545003766978
epoch: 1
Cost:  0.7957285391579009
epoch  1 test cost:  1.6111853201243125 train cost:  0.7912041140452751
epoch: 2
Cost:  0.7939764573471049
epoch  2 test cost:  1.6077956471584107 train cost:  0.7877661610812084
epoch: 3
Cost:  0.7922396431243488
epoch  3 test cost:  1.6044257583718529 train cost:  0.7843332835811461
epoch: 4
Cost:  0.7905103936414055
epoch  4 test cost:  1.6010571621830352 train cost:  0.7809031664104331
epoch: 5
Cost:  0.7887856406424345
epoch  5 test cost:  1.597687186798191 train cost:  0.7774745994710192
epoch: 6
Cost:  0.7870637170709952
epoch  6 test cost:  1.5943160751697842 train cost:  0.7740470223889916
epoch: 7
Cost:  0.7853437224452748
epoch  7 test cost:  1.5909443544918653 train cost:  0.7706201354874966
epoch: 8
Cost:  0.7836251263040247
epoch  8 test cost:  1.5875806381284008 train cost:  0.

c:\Users\rares\Dropbox (MIT)\Documents\documents\mit\research\FulfilmentRules\models.py:81: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  d = torch.tensor(YY[i:i+batch_size:]).float()
c:\Users\rares\Dropbox (MIT)\Documents\documents\mit\research\FulfilmentRules\models.py:82: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input = torch.tensor(X_train[i:i+batch_size,:]).float()


epoch: 32
Cost:  0.7455431581323385
epoch  32 test cost:  1.5162013479664798 train cost:  0.6992491894750213
epoch: 33
Cost:  0.7441648181466989
epoch  33 test cost:  1.513621989142131 train cost:  0.6968543295787402
epoch: 34
Cost:  0.7427968317885205
epoch  34 test cost:  1.5110576940859641 train cost:  0.6944614778664764
epoch: 35
Cost:  0.7414383889795063
epoch  35 test cost:  1.5085104764220496 train cost:  0.692069719989228
epoch: 36
Cost:  0.7400887313400617
epoch  36 test cost:  1.5059681434501064 train cost:  0.6896779685855148
epoch: 37
Cost:  0.7387471687031903
epoch  37 test cost:  1.503447071223741 train cost:  0.6872856214539703
epoch: 38
Cost:  0.737413059315301
epoch  38 test cost:  1.500947601837167 train cost:  0.6849146039774443
epoch: 39
Cost:  0.7360864831444713
epoch  39 test cost:  1.4984279843622768 train cost:  0.6826329623136529
epoch: 40
Cost:  0.7347690436605915
epoch  40 test cost:  1.4959106130956492 train cost:  0.6803543102097671
epoch: 41
Cost:  0.73346

In [32]:
two_cost, two_dec = eval_model(two_stage_forecast, X_test, Y_test, n_samples = 100)

C:\Users\rares\AppData\Local\Temp/ipykernel_24064/2065713499.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(means) * ( 1 + noises)).detach().numpy()


1   0.08469396286613584
2   0.0642890767984432
3   0.0537151035137701
4   0.17768001265045202
5   0.5781910388705429
6   1.3513777627744024
7   1.1634796391653157
8   1.0232593841047022
9   0.930138267447712
10   0.8543970041850791
11   0.7830795594163656
12   0.7202662139598042
13   0.7883667086260603
14   0.8971431691807091
15   0.8386534841784609
16   0.8365100342167173
17   0.798956006840085
18   0.7626236519640037
19   0.8563325455871394
20   0.8152602979697597
21   0.794493007356697
22   0.7932118033635909
23   0.7606397562442805
24   0.7307725167818884
25   0.704582247812657
26   0.7365283466970344
27   0.7107218165973213
28   0.6863335810955824
29   0.6710105124303696
30   0.6574808018488384
31   0.6448659499886245
32   0.6279384900892803
33   0.6548178811339459
34   0.644789029694192
35   0.6328064515356813
36   0.6157012034995645
37   0.605194203673406
38   0.6060853704048607
39   0.6113524430769922
40   0.604596589814139
41   0.5902434262167999
42   0.5807340940592021
43   0

In [33]:
task_cost, task_dec = eval_model(task_forecast, X_test, Y_test, n_samples = 10)

C:\Users\rares\AppData\Local\Temp/ipykernel_24064/2065713499.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return (torch.tensor(means) * ( 1 + noises)).detach().numpy()


1   0.08303524953453867
2   0.06326714125066563
3   0.053156622310327584
4   0.17633781557498654
5   0.5757820105073315
6   1.0592569966066137
7   0.9373529518458038
8   0.8254235897011687
9   0.7569890575559336
10   0.708889236177629
11   0.654930526719149
12   0.6028439288399878
13   0.6804721055179617
14   0.78739993785639
15   0.7362266440456582
16   0.735506200651182
17   0.7067765966824036
18   0.6785197361161777
19   0.7764289039624805
20   0.7393595735259773
21   0.7185866115699103
22   0.7066993135076649
23   0.67786584514784
24   0.6514679642051107
25   0.6297579307019874
26   0.6586009006831541
27   0.6357109869373926
28   0.6140062177047164
29   0.6039389633951546
30   0.5955438906971643
31   0.576790790406353
32   0.5650319559522
33   0.5838260263015017
34   0.5795575757929599
35   0.5710884859010833
36   0.5556786510617506
37   0.5475372316944618
38   0.5350579881372879
39   0.5411372158022318
40   0.5367022547885085
41   0.5240199089511007
42   0.5166883872828185
43   0.

In [16]:
from torch.distributions import Normal

dist = Normal(torch.tensor([0.0]), torch.tensor([1.0]))

In [26]:
two_stage_forecast(X_train)

tensor([[0.0000, 0.0000],
        [0.0000, 0.0000],
        [1.1238, 4.1431],
        [0.0000, 0.0000],
        [0.5212, 2.0895],
        [0.4323, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 1.1545],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.6319],
        [0.5493, 0.0000],
        [1.1667, 2.0707],
        [0.2330, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.0000, 1.1124],
        [1.4225, 4.6869],
        [0.0000, 0.0000],
        [0.0483, 3.8509],
        [0.2649, 0.2840],
        [0.0000, 0.0000],
        [0.2694, 0.2107],
        [0.0000, 0.0000],
        [0.0312, 0.0000],
        [0.0000, 0.5083],
        [0.0000, 2.2433],
        [0.2281, 1.7360],
        [0.0000, 0.0000],
        [0.0000, 0.0000],
        [0.8554, 2.1114],
        [0.0000, 0.0000],
        [1.8996, 2.8543],
        [0.0000, 0.0000],
        [0.6703, 0.9314],
        [0.0

In [8]:
Y_train

tensor([[0.0270, 0.2430],
        [0.0100, 0.0168],
        [0.0191, 0.0100],
        [0.0100, 0.0550],
        [0.0868, 0.0100],
        [0.0100, 0.0100],
        [0.0365, 0.0100],
        [0.0876, 0.0100],
        [0.0238, 0.0820],
        [0.0435, 0.0505],
        [0.0124, 0.5667],
        [0.0214, 0.0529],
        [0.0351, 0.0113],
        [0.0314, 0.0100],
        [0.0100, 0.0100],
        [0.0100, 0.0211],
        [0.0225, 0.3507],
        [0.0193, 0.0606],
        [0.0219, 0.0100],
        [0.0100, 0.0275],
        [0.0100, 0.0100],
        [0.0100, 0.0100],
        [0.0100, 1.3317],
        [0.0100, 0.0100],
        [0.0167, 0.0323],
        [0.0513, 1.7518],
        [0.0239, 0.0100],
        [0.0172, 0.0303],
        [0.0600, 3.3528],
        [0.0449, 0.0532],
        [0.0100, 0.1598],
        [0.0501, 0.0100],
        [0.0100, 0.0154],
        [0.0136, 0.0100],
        [0.0571, 0.0753],
        [0.0718, 0.0100],
        [0.0721, 0.0100],
        [0.0312, 0.0483],
        [0.0